<a href="https://colab.research.google.com/github/BrianCarela/Data-analytics-practice/blob/main/phase1_python_for_data_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Phase 1: Python for Data
### Your Data Analyst Curriculum — Part 2

---

**What you'll practice here:**
- Loading a real dataset with `pandas`
- Exploring, filtering, and grouping tabular data
- Computing stats with `numpy`
- Making your first charts
- One capstone exercise that puts it all together

**Prerequisites:** You've completed the deep dive notebook (Sections 1–5). Variables, lists, loops, comprehensions, and functions should feel comfortable.

---

## 🔧 Section 1: Setup

Import the four libraries. You know what these are.

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print('Libraries loaded.')

Libraries loaded.


## 🐍 Section 2: Quick Python Refresher

A fast recap of what you built in the deep dive — variables, lists, loops, comprehensions, functions. Run these cells to load the functions we'll reuse later.

### 2a. Variables & Types

In [18]:
name = "Pidjin"
age = 30
is_smoker = False
bmi = 24.7

print(f"Hello, {name}. BMI: {bmi}. Smoker: {is_smoker}.")
print(f"Type of bmi: {type(bmi)}")

Hello, Pidjin. BMI: 24.7. Smoker: False.
Type of bmi: <class 'float'>


### 2b. Lists

In [19]:
charges = [1200.50, 3400.00, 875.25, 12000.99, 450.00]

print(f"First: {charges[0]}")
print(f"Last: {charges[-1]}")
print(f"Count: {len(charges)}")
print(f"Average: {sum(charges) / len(charges):.2f}")

First: 1200.5
Last: 450.0
Count: 5
Average: 3585.35


### 2c. Loops

In [20]:
regions = ['northeast', 'northwest', 'southeast', 'southwest']

for i, region in enumerate(regions):
    print(f"{i}: {region}")

0: northeast
1: northwest
2: southeast
3: southwest


### 2d. List Comprehensions

In [21]:
charges = [1200.50, 3400.00, 875.25, 12000.99, 450.00]

charges_with_tax = [c * 1.1 for c in charges]
high_charges = [c for c in charges if c > 1000]

print(f"With tax: {charges_with_tax}")
print(f"Over $1000: {high_charges}")

With tax: [1320.5500000000002, 3740.0000000000005, 962.7750000000001, 13201.089, 495.00000000000006]
Over $1000: [1200.5, 3400.0, 12000.99]


### 2e. Functions

Run this cell — we'll reuse `classify_bmi` later when working with the real dataset.

In [22]:
def classify_bmi(bmi):
    """Returns BMI category as a string."""
    if bmi < 18.5:
        return 'Underweight'
    elif bmi < 25:
        return 'Normal'
    elif bmi < 30:
        return 'Overweight'
    else:
        return 'Obese'

def classify_age(age):
    """Returns age group as a string."""
    if age < 30:
        return 'Young'
    elif age < 60:
        return 'Middle-aged'
    else:
        return 'Senior'

# Quick test
test_bmis = [17.0, 22.5, 27.3, 34.1]
for b in test_bmis:
    print(f"BMI {b} → {classify_bmi(b)}")

BMI 17.0 → Underweight
BMI 22.5 → Normal
BMI 27.3 → Overweight
BMI 34.1 → Obese


### 🏋️ Exercise 1

Write a function called `classify_age` that takes an age and returns:
- `'Young'` if under 30
- `'Middle-aged'` if 30–59
- `'Senior'` if 60 or above

Then test it on ages: `[22, 35, 61, 45, 28]`

*(Note: `classify_age` is already defined above — this exercise is asking you to write it yourself from scratch in the cell below, then compare.)*

In [23]:
# Your code here
def classify_age(age):
  if age < 30:
    return 'Young'
  elif age < 60:
    return 'Middle-aged'
  else:
    return 'Senior'

test_ages = [22, 35, 61, 45, 28]

for a in test_ages:
  print(f"Age {a} is {classify_age(a)}")

Age 22 is Young
Age 35 is Middle-aged
Age 61 is Senior
Age 45 is Middle-aged
Age 28 is Young


---
## 🐼 Section 3: Pandas — Working with Tables

A pandas `DataFrame` is a table — rows and columns, like a spreadsheet. This is the core tool for data analysts.

### 3a. Load the Dataset

We're using the **insurance dataset** — 1,338 rows of patient data including age, BMI, smoker status, region, and yearly medical charges.

Instead of fetching a CSV from a URL that might go down, we load it from seaborn's built-in collection — guaranteed to always work.

In [36]:
# seaborn bundles several classic datasets — no URL needed, no 404 errors
df = sns.load_dataset('healthexp')

# Actually — let's use the proper insurance dataset.
# We'll build it from the OpenML source which is stable.
import urllib.request
import io

url = 'https://raw.githubusercontent.com/BrianCarela/Data-analytics-practice/refs/heads/main/insurance.csv'
try:
    response = urllib.request.urlopen(url)
    # Skip ARFF header lines that start with '@'
    lines = [l.decode('utf-8') for l in response if not l.decode('utf-8').startswith('@')]
    df = pd.read_csv(io.StringIO('\n'.join(lines)))
    print('Loaded from github')
except:
    # Fallback: construct a representative sample inline
    print('URL unavailable — using built-in sample dataset.')
    data = {
        'age':      [19,18,28,33,32,31,46,37,37,60,25,62,23,56,27,19,52,23,56,30,
                     60,30,18,34,37,59,63,55,23,31,22,18,34,20,19,22,57,25,44,26,
                     47,26,19,23,33,35,53,35,47,34],
        'sex':      ['female','male','male','male','male','female','female','female','male','female',
                     'male','female','male','female','male','female','male','male','male','male',
                     'female','male','male','female','male','female','male','male','female','female',
                     'male','female','male','female','male','male','female','male','male','male',
                     'male','female','male','female','male','male','male','female','male','female'],
        'bmi':      [27.9,33.77,33.0,22.705,28.88,25.74,33.44,27.74,29.83,25.84,
                     26.22,26.29,34.4,27.72,42.13,24.6,26.315,17.385,28.0,36.85,
                     32.4,23.2,28.31,31.92,34.1,27.72,23.085,32.775,28.05,33.4,
                     17.19,18.335,34.105,28.6,26.315,23.75,45.36,22.515,30.715,29.7,
                     34.01,33.0,34.43,36.765,27.36,30.115,22.135,22.8,27.5,31.0],
        'children': [0,1,3,0,0,0,1,3,2,0,0,0,0,0,0,1,0,0,0,1,
                     0,0,0,0,2,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,
                     1,0,0,0,0,1,0,0,1,0],
        'smoker':   ['yes','no','no','no','no','no','no','no','no','no',
                     'no','yes','no','no','yes','no','no','no','yes','yes',
                     'no','no','no','no','no','no','yes','no','no','no',
                     'no','no','yes','no','no','no','no','no','no','yes',
                     'no','no','no','no','no','no','no','no','yes','no'],
        'region':   ['southwest','southeast','southeast','northwest','northwest','southeast','southeast','northwest','northeast','northwest',
                     'northeast','southeast','southwest','northeast','southeast','northeast','northeast','northeast','southwest','southeast',
                     'southwest','southwest','southeast','northeast','northeast','northeast','northwest','southeast','southeast','southwest',
                     'northeast','southeast','southeast','southwest','northeast','northeast','southeast','northwest','southeast','northeast',
                     'northwest','southeast','southwest','southeast','northeast','southwest','northeast','northeast','southeast','southwest'],
        'charges':  [16884.924,1725.5523,4449.462,21984.47061,3866.8552,3756.6216,8240.5896,7281.5056,6406.4107,28923.13692,
                     2721.3208,27808.7251,1826.843,11090.7178,39611.7577,1837.237,10797.3362,2395.17155,10602.385,5272.1758,
                     28923.13692,2395.17155,2352.9682,5267.25,7258.6688,13457.9612,23045.56616,10226.2842,2395.17155,6435.609,
                     1826.843,1631.8212,36837.467,1534.3208,1825.35,2395.17155,42303.6924,2198.18985,18963.17195,2302.3,
                     8083.9198,4564.191,8606.2171,26187.91947,2414.7995,10791.9602,4439.9553,11244.3765,20709.01894,4718.7]
    }
    df = pd.DataFrame(data)

print(f'Shape: {df.shape}')  # (rows, columns)
df.head()

Loaded from github
Shape: (1338, 7)


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


### 3b. First Look — Always Do This First

Every time you get a new dataset, run these three things before anything else. This is the standard opening move for any data analyst.

In [31]:
# 1. Column names, data types, and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1339 entries, 0 to 1338
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   age       1339 non-null   object
 1   sex       1339 non-null   object
 2   bmi       1339 non-null   object
 3   children  1339 non-null   object
 4   smoker    1339 non-null   object
 5   region    1339 non-null   object
 6   charges   1339 non-null   object
dtypes: object(7)
memory usage: 73.4+ KB


In [32]:
# 2. Summary statistics on all numeric columns
# count, mean, std, min, 25th percentile, median, 75th percentile, max
df.describe()

,age,sex,bmi,children,smoker,region,charges
count,1339,1339,1339,1339,1339,1339,1339
unique,48,3,549,7,3,5,1338
top,18,male,32.3,0,no,southeast,1639.5631
freq,69,676,13,574,1064,364,2


In [33]:
# 3. Check for missing values
# In real datasets, missing data is everywhere — always check
df.isnull().sum()

,0
age,0
sex,0
bmi,0
children,0
smoker,0
region,0
charges,0


### 3c. Selecting Data

Two syntaxes to know:
- `df['column']` — single column → returns a **Series** (like a labeled list)
- `df[['col1', 'col2']]` — multiple columns → returns a **DataFrame**

In [37]:
# Single column — Series
ages = df['age']
print(ages.head())
print(f"\nAverage age: {ages.mean():.1f}")
print(f"Min: {ages.min()}, Max: {ages.max()}")

0    19
1    18
2    28
3    33
4    32
Name: age, dtype: int64

Average age: 39.2
Min: 18, Max: 64


In [35]:
# Multiple columns — DataFrame
subset = df[['age', 'bmi', 'charges']]
subset.head()

,age,bmi,charges
0,age,bmi,charges
1,19,27.9,16884.924
2,18,33.77,1725.5523
3,28,33,4449.462
4,33,22.705,21984.47061


### 3d. Filtering Rows

This is SQL's `WHERE` clause. You put a condition inside `df[...]` and only the matching rows come back.

**Important:** Use `&` for AND and `|` for OR — not `and`/`or`. And wrap each condition in `()`.

In [ ]:
# Smokers only
smokers = df[df['smoker'] == 'yes']
non_smokers = df[df['smoker'] == 'no']

print(f"Smoker count: {len(smokers)}")
print(f"Average charge (smokers):     ${smokers['charges'].mean():,.2f}")
print(f"Average charge (non-smokers): ${non_smokers['charges'].mean():,.2f}")
print(f"Smokers pay {smokers['charges'].mean() / non_smokers['charges'].mean():.1f}x more on average")

In [ ]:
# Multiple conditions: smokers AND age over 40
# Each condition wrapped in () — required when using & or |
older_smokers = df[(df['smoker'] == 'yes') & (df['age'] > 40)]
print(f"Older smokers (40+): {len(older_smokers)} people")
print(f"Their average charge: ${older_smokers['charges'].mean():,.2f}")
older_smokers.head()

### 3e. GroupBy — Summarizing by Category

This is SQL's `GROUP BY`. The most important pandas operation for data analysts.

Pattern: `df.groupby('category_column')['value_column'].aggregation()`

In [ ]:
# Average charges by region — sorted highest to lowest
df.groupby('region')['charges'].mean().sort_values(ascending=False)

In [ ]:
# Multiple stats at once using .agg()
df.groupby('smoker')['charges'].agg(['mean', 'median', 'min', 'max', 'count'])

In [ ]:
# Group by TWO columns — creates a cross-tab view
# .unstack() pivots the inner group into columns for easier reading
df.groupby(['region', 'smoker'])['charges'].mean().round(2).unstack()

### 3f. Adding New Columns

`.apply()` runs a function on every row of a column — this is how you bring your custom functions into pandas.

In [ ]:
# Apply classify_bmi to every value in the 'bmi' column
df['bmi_category'] = df['bmi'].apply(classify_bmi)

# value_counts() shows how many rows fall into each category
print('BMI category distribution:')
print(df['bmi_category'].value_counts())

In [ ]:
# Now we can groupby our new column
df.groupby('bmi_category')['charges'].mean().sort_values(ascending=False)

### 🏋️ Exercise 2

Use the `classify_age` function (defined in Section 2e) to:
1. Add an `age_group` column to `df` using `.apply()`
2. Find the average charges for each age group using `.groupby()`
3. Print a sentence saying which age group has the highest average charges

*Hint: `.idxmax()` on a Series returns the index label of the maximum value.*

In [ ]:
# Your code here


---
## 🔢 Section 4: NumPy — Math & Stats

NumPy works on arrays of numbers — fast, vectorized math. Pandas uses it internally.

### 4a. Arrays vs Lists

In [ ]:
charges_list = [1200.50, 3400.00, 875.25, 12000.99, 450.00]
charges_array = np.array(charges_list)

# Math on an array applies to EVERY element at once — no loop needed
print('Array * 1.1:')
print(charges_array * 1.1)

# Boolean mask — True/False for each element
print('\nArray > 1000:')
print(charges_array > 1000)

# Use the mask to filter — only keeps True positions
print('\nFiltered (> 1000):')
print(charges_array[charges_array > 1000])

### 4b. Descriptive Stats

These are the stats you'll cite constantly as an analyst. Know them cold.

In [ ]:
# .values converts a pandas Series to a numpy array
charges = df['charges'].values

print(f"Count:    {len(charges)}")
print(f"Mean:     ${np.mean(charges):,.2f}")
print(f"Median:   ${np.median(charges):,.2f}")
print(f"Std Dev:  ${np.std(charges):,.2f}")
print(f"Min:      ${np.min(charges):,.2f}")
print(f"Max:      ${np.max(charges):,.2f}")
print()
print(f"25th percentile: ${np.percentile(charges, 25):,.2f}")
print(f"75th percentile: ${np.percentile(charges, 75):,.2f}")
iqr = np.percentile(charges, 75) - np.percentile(charges, 25)
print(f"IQR (middle 50% spread): ${iqr:,.2f}")

**⚠️ Analyst Insight — Mean vs Median:**

The mean is higher than the median. That means the distribution is *right-skewed* — a small number of very high charges are pulling the average up. The median is more representative of a typical patient.

If you reported only the mean to a non-technical audience, you'd be overstating what most people actually pay. This is one of the most common mistakes in data storytelling.

---
## 📈 Section 5: First Charts

The goal here is to *see* the data you've been computing. Don't worry about memorizing the syntax — focus on what the chart reveals.

### 5a. Distribution of Charges

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df['charges'], bins=40, color='steelblue', edgecolor='white')
plt.axvline(df['charges'].mean(),
            color='red', linestyle='--',
            label=f"Mean: ${df['charges'].mean():,.0f}")
plt.axvline(df['charges'].median(),
            color='orange', linestyle='--',
            label=f"Median: ${df['charges'].median():,.0f}")
plt.title('Distribution of Insurance Charges')
plt.xlabel('Charges ($)')
plt.ylabel('Number of People')
plt.legend()
plt.tight_layout()
plt.show()
print('The long right tail shows a small group of very high-cost patients pulling the mean up.')

### 5b. Smokers vs Non-Smokers

A **box plot** shows the median, the middle 50% of data (the box), and outliers (the dots). Great for comparing groups.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='smoker', y='charges', palette='Set2')
plt.title('Insurance Charges: Smokers vs Non-Smokers')
plt.xlabel('Smoker')
plt.ylabel('Charges ($)')
plt.tight_layout()
plt.show()
print('The entire smoker box sits above the non-smoker box — smoking is the biggest cost driver.')

### 5c. Age vs Charges

A **scatter plot** shows the relationship between two numeric variables. Color adds a third dimension.

In [ ]:
plt.figure(figsize=(10, 5))
sns.scatterplot(data=df, x='age', y='charges',
                hue='smoker', alpha=0.6, palette='Set1')
plt.title('Age vs Charges (by Smoker Status)')
plt.xlabel('Age')
plt.ylabel('Charges ($)')
plt.tight_layout()
plt.show()
print('Notice the two distinct bands for smokers — charges increase with age, but smokers start higher.')

---
## 🏁 Section 6: Milestone Exercise

This is your Phase 1 capstone. Use everything you've learned.

**The question:** *Which combination of factors is most associated with high insurance charges?*

1. What is the average charge for someone who is a smoker AND has an `'Obese'` BMI?
2. Compare that to a non-smoker with a `'Normal'` BMI.
3. How many times more expensive is the high-risk group?
4. Write 2–3 sentences (as a print statement) describing your finding as if presenting to a manager.

*Analyst mindset: find a number, give it meaning, communicate it clearly.*

In [ ]:
# Your code here


---
## ✅ Phase 1 Complete

When you finish the milestone, you're ready for **Phase 2: EDA & Visualization.**

Phase 2 covers:
- Reading and explaining every part of a chart
- Distributions, skew, and outlier detection
- Correlation matrices — where your old notebook left off
- Writing analyst-style observations under each chart
- Telling a data story from scratch

**Push this to GitHub and send me the raw link when you finish the milestone. 🚀**